In [1]:
import os
import numpy as np
import pandas as pd
import torch
from pathlib import Path

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

torch.set_grad_enabled(False)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True



In [2]:
test_data = pd.read_csv(
    "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv"
)
test_data




,essay_id,full_text
0,d550b2d,The face was not created by aliens because the...
1,0c10954,Hello my name is Luke Bomberger and I was seag...
2,ef04816,The technology to read the emotional expressio...
3,88ab8d9,Do you ever wonder how many soomething is or h...
4,cee8c0f,This story is about discovering mars. Witch is...
...,...,...
1726,d8c9e3d,The devlopment of driverless cars should not b...
1727,6c3e2b9,this story is about how hard it is about how h...
1728,743f56b,Do you think the author did a great job on sug...
1729,03630c7,Have you ever heard of the Seagoing Cowboys pr...


In [3]:
# Fix: load an offline model from local Kaggle input (no internet, no HF cache guaranteed).
# If none exists, fall back to a deterministic baseline so we still write a valid submission.csv.
def _looks_like_hf_model_dir(p: Path) -> bool:
    if not p.is_dir():
        return False
    # Prefer safetensors if available, but allow pytorch_model.bin too.
    has_config = (p / "config.json").is_file()
    has_weights = (p / "model.safetensors").is_file() or (
        p / "pytorch_model.bin"
    ).is_file()
    # Tokenizer can be split across files; these are common indicators.
    has_tokenizer = any(
        (p / fn).is_file()
        for fn in ["tokenizer.json", "vocab.txt", "merges.txt", "spiece.model"]
    )
    return has_config and has_weights and has_tokenizer


def _find_local_model_dirs(search_roots):
    candidates = []
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue
        # Keep search bounded for time; depth-limited via rglob but filtered by filename hits.
        for cfg in root.rglob("config.json"):
            p = cfg.parent
            if _looks_like_hf_model_dir(p):
                candidates.append(p)
    # Deterministic order
    candidates = sorted(set(candidates), key=lambda x: str(x))
    return candidates


def _try_load_from_local_dir(model_dir: Path):
    tok = AutoTokenizer.from_pretrained(model_dir, local_files_only=True, use_fast=True)
    mdl = AutoModelForSequenceClassification.from_pretrained(
        model_dir, local_files_only=True
    )
    return tok, mdl


tokenizer = None
model = None
MODEL_PATH = None
last_err = None

search_roots = [
    "/kaggle/input",
    "/kaggle/data",
]

local_model_dirs = _find_local_model_dirs(search_roots)

# Try local model directories first (offline-safe).
for p in local_model_dirs:
    try:
        tokenizer, model = _try_load_from_local_dir(p)
        MODEL_PATH = p
        break
    except Exception as e:
        last_err = e
        tokenizer, model, MODEL_PATH = None, None, None

# If we loaded a model, ensure 6 labels (core logic preserved: argmax over 6 classes -> +1 score).
if model is not None and tokenizer is not None:
    if getattr(model.config, "num_labels", None) != 6:
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_PATH,
            num_labels=6,
            ignore_mismatched_sizes=True,
            local_files_only=True,
        )
    model.eval()
    print("Loaded local offline model:", str(MODEL_PATH))
    print("Model num_labels:", getattr(model.config, "num_labels", None))
else:
    print(
        "WARNING: No local Transformers checkpoint found under /kaggle/input or /kaggle/data.\n"
        "Falling back to deterministic baseline (predict global mean score from train.csv)."
    )
    if last_err is not None:
        print("Last model load error:", repr(last_err))



Falling back to deterministic baseline (predict global mean score from train.csv).


In [4]:
MAX_LEN = 1536

tok_kwargs = dict(
    truncation=True,
    max_length=MAX_LEN,
    padding=False,  # collator will pad dynamically
    return_attention_mask=True,
    return_token_type_ids=False,
)

os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")


class RawTextDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx]


test_dataset = RawTextDataset(test_data["full_text"].tolist())


class TokenizeCollator:
    def __init__(self, tokenizer, tok_kwargs, pad_to_multiple_of=None):
        self.tokenizer = tokenizer
        self.tok_kwargs = {k: v for k, v in tok_kwargs.items() if k != "padding"}
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, batch_texts):
        enc = self.tokenizer(
            batch_texts,
            **self.tok_kwargs,
            padding="longest",
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}




In [5]:
# Fix: only build GPU loader / move model if a transformer model actually loaded.
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")

test_loader = None
if model is not None:
    model.to(device)

    if hasattr(model, "config"):
        try:
            model.config.use_cache = False
        except Exception:
            pass

    if use_cuda and os.environ.get("DISABLE_TORCH_COMPILE", "0") != "1":
        try:
            model = torch.compile(model, mode="reduce-overhead", fullgraph=False)
        except Exception:
            pass

    pad_to_multiple_of = 8 if use_cuda else None
    collate_fn = TokenizeCollator(
        tokenizer=tokenizer,
        tok_kwargs=tok_kwargs,
        pad_to_multiple_of=pad_to_multiple_of,
    )

    if use_cuda:
        num_workers = min(4, os.cpu_count() or 2)
        prefetch_factor = 4
        persistent_workers = True
    else:
        num_workers = 0
        prefetch_factor = None
        persistent_workers = False

    dl_kwargs = dict(
        dataset=test_dataset,
        batch_size=(64 if use_cuda else 8),
        shuffle=False,
        num_workers=num_workers,
        pin_memory=use_cuda,
        collate_fn=collate_fn,
        persistent_workers=persistent_workers,
    )
    if num_workers > 0:
        dl_kwargs["prefetch_factor"] = prefetch_factor

    test_loader = DataLoader(**dl_kwargs)

    if use_cuda:
        try:
            model = model.to(memory_format=torch.channels_last)
        except Exception:
            pass



In [6]:
# Fix: if no model is available offline, produce a stable baseline prediction to ensure valid submission.
if model is None:
    train_path = "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/train.csv"
    train_df = pd.read_csv(train_path, usecols=["score"])
    mean_score = float(train_df["score"].mean())
    baseline_score = int(np.clip(np.rint(mean_score), 1, 6))
    pred_classes = np.full(len(test_dataset), baseline_score - 1, dtype=np.int64)
else:
    pred_classes = np.empty(len(test_dataset), dtype=np.int64)

    model.eval()
    offset = 0

    with torch.inference_mode():
        for batch in test_loader:
            if use_cuda:
                batch["input_ids"] = batch["input_ids"].to(device, non_blocking=True)
                batch["attention_mask"] = batch["attention_mask"].to(
                    device, non_blocking=True
                )
            else:
                batch["input_ids"] = batch["input_ids"].to(device)
                batch["attention_mask"] = batch["attention_mask"].to(device)

            outputs = model(**batch)
            batch_pred = torch.argmax(outputs.logits, dim=-1)
            batch_pred_np = batch_pred.cpu().numpy().astype(np.int64, copy=False)

            bs = batch_pred_np.shape[0]
            pred_classes[offset : offset + bs] = batch_pred_np
            offset += bs

    assert offset == len(test_dataset)



In [7]:
predicted_scores = (pred_classes + 1).clip(1, 6).astype("int32")



In [8]:
submission = pd.DataFrame(
    {"essay_id": test_data["essay_id"].values, "score": predicted_scores}
)

submission.to_csv("submission.csv", index=False)
submission



,essay_id,score
0,d550b2d,3
1,0c10954,3
2,ef04816,3
3,88ab8d9,3
4,cee8c0f,3
...,...,...
1726,d8c9e3d,3
1727,6c3e2b9,3
1728,743f56b,3
1729,03630c7,3


In [9]:
assert submission.shape[0] == test_data.shape[0]
assert list(submission.columns) == ["essay_id", "score"]
assert submission["score"].between(1, 6).all()
print("Wrote submission.csv with shape:", submission.shape)

Wrote submission.csv with shape: (1731, 2)
